In [ ]:
# Deepfake Detector using OpenCV and a Pre-trained Model

import cv2
import numpy as np
from keras.preprocessing import image
from keras.models import load_model

# Load the pre-trained deepfake detection model
try:
    model = load_model('deepfake_detector_model.h5')
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# Function to process and predict deepfakes
def predict_deepfake(video_path):
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Unable to open video file {video_path}")
        return
    
    print("Processing video... Press 'q' to quit.")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("End of video or error reading frame.")
            break
            
        try:
            # Preprocess frame for the model
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img = image.img_to_array(img)
            img = np.expand_dims(img, axis=0)
            img = img / 255.0
            
            # Make prediction
            prediction = model.predict(img)
            
            # Determine label
            label = 'Fake' if prediction[0][0] > 0.5 else 'Real'
            
            # Display prediction on the frame
            cv2.putText(frame, label, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
            
            # Show frame with prediction label
            cv2.imshow('Deepfake Detector', frame)
        except Exception as e:
            print(f"Error processing frame: {e}")
            break
        
        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Exiting...")
            break
            
    # Release resources
    cap.release()
    cv2.destroyAllWindows()

# Example usage
video_path = 'test_video.mp4'
predict_deepfake(video_path)